# 01_bronze_to_silver: BikeStores
Reads raw Bronze tables, standardises types, removes duplicates, handles nulls,
runs data quality rules, quarantines bad rows and logs row counts per table.

No credentials in this notebook: storage is reached through the Unity Catalog external location.

In [0]:
from datetime import datetime
from pyspark.sql import functions as F

c = F.col

STORAGE   = "cdestorageacc"
CONTAINER = "onpremsqlserver"
BASE      = f"abfss://{CONTAINER}@{STORAGE}.dfs.core.windows.net"
BRONZE    = BASE                  # Bronze CSVs sit at the container root: <schema>/<table>/<table>.csv
SILVER    = f"{BASE}/silver"
REJECTS   = f"{BASE}/silver_rejects"
LOGS      = f"{BASE}/logs"
BRONZE_FORMAT = "csv"             # Bronze is CSV; Silver and Gold stay Delta

def bronze_path(table):
    schema, name = table.split(".")
    return f"{BRONZE}/{schema}/{name}"   # adjust to your Data Factory sink folder pattern

## Cleaning and data quality rules per table

In [0]:
EMAIL = r"^[^@\s]+@[^@\s]+\.[^@\s]+$"

SPEC = {
    "sales.customers": dict(
        keys=["customer_id"],
        types={"customer_id": "int"},
        defaults={"phone": "Unknown"},
        rules={
            "customer_id_not_null": c("customer_id").isNotNull(),
            "valid_email": c("email").rlike(EMAIL),
        }),
    "sales.stores": dict(
        keys=["store_id"],
        types={"store_id": "int"},
        defaults={},
        rules={"store_id_not_null": c("store_id").isNotNull()}),
    "sales.staffs": dict(
        keys=["staff_id"],
        types={"staff_id": "int", "active": "int", "store_id": "int", "manager_id": "int"},
        defaults={},
        rules={"staff_id_not_null": c("staff_id").isNotNull()}),
    "sales.orders": dict(
        keys=["order_id"],
        types={"order_id": "int", "customer_id": "int", "order_status": "int",
               "order_date": "date", "required_date": "date", "shipped_date": "date",
               "store_id": "int", "staff_id": "int"},
        defaults={},
        rules={
            "valid_order_date": c("order_date").between(F.lit("2000-01-01").cast("date"), F.current_date()),
            "valid_status": c("order_status").isin(1, 2, 3, 4),
            "shipped_not_before_order": c("shipped_date").isNull() | (c("shipped_date") >= c("order_date")),
        }),
    "sales.order_items": dict(
        keys=["order_id", "item_id"],
        types={"order_id": "int", "item_id": "int", "product_id": "int", "quantity": "int",
               "list_price": "decimal(10,2)", "discount": "decimal(4,2)"},
        defaults={},
        rules={
            "quantity_positive": c("quantity") > 0,
            "price_non_negative": c("list_price") >= 0,
            "discount_between_0_and_1": c("discount").between(0, 1),
        }),
    "production.products": dict(
        keys=["product_id"],
        types={"product_id": "int", "brand_id": "int", "category_id": "int",
               "model_year": "int", "list_price": "decimal(10,2)"},
        defaults={},
        rules={
            "name_not_null": c("product_name").isNotNull(),
            "price_non_negative": c("list_price") >= 0,
            "valid_model_year": c("model_year") >= 1990,
        }),
    "production.categories": dict(
        keys=["category_id"], types={"category_id": "int"}, defaults={},
        rules={"category_id_not_null": c("category_id").isNotNull()}),
    "production.brands": dict(
        keys=["brand_id"], types={"brand_id": "int"}, defaults={},
        rules={"brand_id_not_null": c("brand_id").isNotNull()}),
    "production.stocks": dict(
        keys=["store_id", "product_id"],
        types={"store_id": "int", "product_id": "int", "quantity": "int"},
        defaults={},
        rules={"quantity_non_negative": c("quantity") >= 0}),
}

## Process each table

In [0]:
def process(table, spec):
    reader = spark.read.format(BRONZE_FORMAT)
    if BRONZE_FORMAT == "csv":
        # read everything as strings; the casting step below standardises the types
        reader = reader.option("header", True).option("nullValue", "NULL").option("inferSchema", False)
    df = reader.load(f"{BASE}/Bronze/{table.replace('.', '/')}")
    rows_read = df.count()

    # 1. standardise data types
    for col_name, dtype in spec["types"].items():
        df = df.withColumn(col_name, c(col_name).cast(dtype))

    # 2. trim strings, turn blanks into nulls
    for f in df.schema.fields:
        if f.dataType.simpleString() == "string":
            df = df.withColumn(f.name, F.when(F.trim(c(f.name)) == "", None).otherwise(F.trim(c(f.name))))

    # 3. handle missing values with agreed defaults
    if spec["defaults"]:
        df = df.fillna(spec["defaults"])

    # 4. remove duplicate records on the business key
    deduped = df.dropDuplicates(spec["keys"])
    rows_after_dedupe = deduped.count()

    # 5. data quality checks; a failed rule marks the row as rejected
    reasons = F.concat_ws(
        "; ",
        *[F.when(~F.coalesce(cond, F.lit(False)), F.lit(rule)) for rule, cond in spec["rules"].items()],
    )
    checked = deduped.withColumn("_reject_reason", reasons)
    good = checked.filter(c("_reject_reason") == "").drop("_reject_reason")
    bad = checked.filter(c("_reject_reason") != "").withColumn("_rejected_at", F.current_timestamp())

    rows_written = good.count()
    rows_rejected = bad.count()

    folder = table.replace(".", "/")
    good.write.format("delta").mode("overwrite").option("overwriteSchema", "true").save(f"{SILVER}/{folder}")
    bad.write.format("delta").mode("overwrite").option("overwriteSchema", "true").save(f"{REJECTS}/{folder}")

    return dict(
        run_ts=datetime.utcnow().isoformat(timespec="seconds"),
        notebook="01_bronze_to_silver",
        table=table,
        rows_read=rows_read,
        duplicates_removed=rows_read - rows_after_dedupe,
        rows_rejected=rows_rejected,
        rows_written=rows_written,
    )

log = []
for table, spec in SPEC.items():
    entry = process(table, spec)
    log.append(entry)
    print(f"{table}: read={entry['rows_read']} dupes={entry['duplicates_removed']} "
          f"rejected={entry['rows_rejected']} written={entry['rows_written']}")

/home/spark-d6b37f29-f467-4f04-b9e2-ea/.ipykernel/63/command-6370259885002370-1476235329:43: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  run_ts=datetime.utcnow().isoformat(timespec="seconds"),


sales.customers: read=1445 dupes=0 rejected=0 written=1445


/home/spark-d6b37f29-f467-4f04-b9e2-ea/.ipykernel/63/command-6370259885002370-1476235329:43: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  run_ts=datetime.utcnow().isoformat(timespec="seconds"),


sales.stores: read=3 dupes=0 rejected=0 written=3
sales.staffs: read=10 dupes=0 rejected=0 written=10
sales.orders: read=1615 dupes=0 rejected=0 written=1615
sales.order_items: read=4722 dupes=0 rejected=0 written=4722
production.products: read=321 dupes=0 rejected=0 written=321
production.categories: read=7 dupes=0 rejected=0 written=7
production.brands: read=9 dupes=0 rejected=0 written=9
production.stocks: read=939 dupes=0 rejected=0 written=939


## Persist the run log

In [0]:
log_df = spark.createDataFrame(log)
log_df.write.format("delta").mode("append").save(f"{LOGS}/silver_run_log")
display(log_df)

duplicates_removed,notebook,rows_read,rows_rejected,rows_written,run_ts,table
0,01_bronze_to_silver,1445,0,1445,2026-11-12T08:32:54,sales.customers
0,01_bronze_to_silver,3,0,3,2026-11-12T08:32:57,sales.stores
0,01_bronze_to_silver,10,0,10,2026-11-12T08:33:01,sales.staffs
0,01_bronze_to_silver,1615,0,1615,2026-11-12T08:33:05,sales.orders
0,01_bronze_to_silver,4722,0,4722,2026-11-12T08:33:08,sales.order_items
0,01_bronze_to_silver,321,0,321,2026-11-12T08:33:12,production.products
0,01_bronze_to_silver,7,0,7,2026-11-12T08:33:15,production.categories
0,01_bronze_to_silver,9,0,9,2026-11-12T08:33:19,production.brands
0,01_bronze_to_silver,939,0,939,2026-11-12T08:33:22,production.stocks
